<a href="https://colab.research.google.com/github/maxkolos/fin_llm/blob/main/multi_agent_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connect to GPU runtime

1. Click the **Runtime** drop-down menu in the upper-right corner and select **Change runtime type**.

![Runtime settings](https://raw.githubusercontent.com/maxkolos/fin_llm/refs/heads/main/images/runtime_settings.png)



2. Select **T4 GPU** from the Hardware accelerator list and click **Save**.

![Runtime type](https://raw.githubusercontent.com/maxkolos/fin_llm/refs/heads/main/images/runtime_type.png)



3. In the top menu bar, select **Run all**.

![Run all](https://raw.githubusercontent.com/maxkolos/fin_llm/refs/heads/main/images/run_all.png)

# Prerequisites setup

This section installs all necessary dependencies.

**CRITICAL NOTE:** Accept a restart prompt once the section has completed (or restart the runtime manually: Runtime -> Restart session).


In [ ]:
# @title 1. Install Python packages
# Note: Avoid using '-q' or similar quiet flags. Full logs are required for
# Colab to detect a dependency updates warning and prompt a session restart.
# Note: don't add -q or so - it prevents a warning that rightfully triggers a restart session dialog.
!pip install transformers accelerate autoawq langgraph langchain-huggingface gptqmodel

print("[ACTION REQUIRED] Restart the runtime session to use newly installed packages and click 'Run All' once again")

# Main workflow

Once the prerequisites are installed and the runtime has restarted, run this section to test the main workflow:

1. *Option 1 (Easiest)*: Click **Run all** in the top toolbar or in the main menu **Runtime -> Run all** (re-running [the prerequisites section](#scrollTo=csFOiS2Vm2Ao) is safe and fast).
2. *Option 2 (Fastest)*: Open the **Table of Contents** in the left sidebar and click the play/triangle button ▷ next to "Main workflow".
3. *Option 3 (Manual)*: Execute the code manually cell by cell [below](#scrollTo=cKKirfIGsLIn).  

Once execution completes, navigate down to [the final cell](#scrollTo=OEtHCElssLIt) and enter your custom prompt to test the multi-agent system.


In [ ]:
%%time
# @title 2. Download an LLM model
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    from langchain_huggingface import HuggingFacePipeline
except ImportError:
    print("[FATAL] Cannot do all imports because the session was not restarted after installations.")
    print("Triggering a runtime session restart...")
    print("You can ignore the popup 'Your session crashed for an unknown reason.'")
    print("[ACTION REQUIRED] Re-run this cell and below after session restart")

    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

    # Prevents printing the exception info.
    get_ipython()._showtraceback = lambda *args, **kwargs: None
    # Prevents executing the cell further while do_shutdown is working.
    import sys
    sys.exit()

# 14B is maximum for Colab free tier, 32B works in Kaggle's free tier.
model_path = "Qwen/Qwen2.5-14B-Instruct-AWQ"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True
)

In [ ]:
%%time
# @title Build a multi-agent system
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Shared State of Agents
class AgentState(TypedDict):
    news: str          # Input market news
    analysis: str      # Analyst recommendation
    feedback : str     # Critic feedback
    iterations: int    # Validation loop counter

def call_local_llm(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Node 1: Analyst (Generates and refines recommendation)
def analyst_node(state: AgentState):
    iterations = state.get("iterations") + 1

    prompt = f"Analyze these stock market news and give a buy/sell recommendation:\n{state['news']}"
    if "feedback" in  state:
        prompt += (
            f"\n\nTake into account critic's feedback to your previous analysis.\n\n"
            f"Your previous analysis: {state['analysis']}\n\n"
            f"Critique to address: {state['feedback']}\n\n"
        )
    response = call_local_llm(prompt)
    return {"analysis": response, "iterations": iterations}

# Node 2: Critic (Checks risks)
def critic_node(state: AgentState):
    prompt = f"Critique briefly this financial analysis for risks and missing facts:\n{state['analysis']}"
    response = call_local_llm(prompt)
    return {"feedback": response}

# Routing function (Exit condition)
def should_continue(state: AgentState):
    if state["iterations"] >= 2:
        return "end"
    return "continue"

# Graph assembly
workflow = StateGraph(AgentState)
# Add nodes
workflow.add_node("analyst", analyst_node)
workflow.add_node("critic", critic_node)
# Add edges
workflow.add_edge(START, "analyst")
workflow.add_conditional_edges(
    "analyst",
    should_continue,
    {
        "continue": "critic",
        "end": END
    }
)
workflow.add_edge("critic", "analyst")

app = workflow.compile()

print("Graph structure:")
from IPython.display import Image, display
display(Image(data=app.get_graph().draw_mermaid_png()))

In [ ]:
%%time
# @title 4. Send a prompt to the multi-agent system {"single-column":true}
prompt = "Apple reports record Q3 earnings, but supply chain issues trigger 2% drop in after-hours trading." # @param {"type":"string","placeholder":"Enter your prompt"}

inputs = {
    "news": prompt,
    "iterations": 0
}

result = app.invoke(inputs)
print(result["analysis"])